# Image and EDS Point Spectra

Acquire an image and collect EDS spectra at selected beam positions.


### Run the servers

Make sure you are on the VPN and the AutoScript server is running. Then start the asyncroscopy Tango servers from the repository root:

```bash
uv run scripts/run_servers.py
```


### Imports


In [ ]:
import os
import json
import time

import tango
import numpy as np
import matplotlib.pyplot as plt
from tiled.client import from_uri

import sidpy

%matplotlib ipympl


### Ping servers


In [ ]:
DB_HOST = "10.46.217.241"
DB_PORT = 9094

os.environ["TANGO_HOST"] = f"{DB_HOST}:{DB_PORT}"

server_names = ['stage', 'scan', 'eds', 'camera', 'data', 'microscope']

for name in server_names:
    device_name = f"asyncroscopy/{name}/default"
    proxy = tango.DeviceProxy(device_name)
    proxy.ping()
    print(device_name, proxy.state())


### Connect to devices


In [ ]:
scan = tango.DeviceProxy("asyncroscopy/scan/default")
microscope = tango.DeviceProxy("asyncroscopy/microscope/default")
data = tango.DeviceProxy("asyncroscopy/data/default")
eds_proxy = tango.DeviceProxy("asyncroscopy/eds/default")

# Backward-compatible aliases used by the workflow cells below.
mic_proxy = microscope
microscope_proxy = microscope

for proxy in (scan, microscope, data, eds_proxy):
    proxy.set_timeout_millis(120_000)

print("scan      :", scan.state())
print("microscope:", microscope.state())
print("data      :", data.state())
print("eds       :", eds_proxy.state())


### Start Tiled data server


In [ ]:
TILED_HOST = "10.46.217.241"
TILED_PORT = 9091
save_path = "D:/microscopedata/tiled/ahoust17/2026_05_22_AtomFab/"

data.host = TILED_HOST
data.port = TILED_PORT
data.save_path = save_path

if str(data.tiled_server).lower() != "yes":
    print("Tiled server is not responding; starting it from the DATA device...")
    config = json.loads(data.start_tiled_server())
else:
    print("Tiled server is already running.")
    config = json.loads(data.get_config())

print(json.dumps(config, indent=2))

client = from_uri(config.get("uri", f"http://{TILED_HOST}:{TILED_PORT}"))
print("Tiled keys:", list(client))


### Acquire a reference image


In [ ]:
scan.Activate(["haadf"])
scan.dwell_time = 10e-6
scan.imsize = 512
scan.scan_region = [0, 0, 1, 1]

key = microscope.acquire_scanned_image()
node = client[key]
overview_image = np.asarray(node.read())
image_metadata = dict(node.metadata)

print("Tiled key     :", key)
print("Image metadata:", image_metadata)
print("Image shape   :", overview_image.shape)


In [ ]:
fig, ax = plt.subplots(figsize=(6, 6))
ax.imshow(overview_image, cmap="gray", interpolation="none")
ax.set_title("Reference HAADF image")
ax.axis("off")
plt.tight_layout()


### Choose random beam positions


In [ ]:
rng = np.random.default_rng(7)
n_points = 5
beam_points = rng.uniform(0.1, 0.9, size=(n_points, 2))

fig, ax = plt.subplots(figsize=(6, 6))
ax.imshow(overview_image, cmap="gray", interpolation="none")
height, width = overview_image.shape
ax.scatter(beam_points[:, 0] * (width - 1), beam_points[:, 1] * (height - 1), c="tab:red", s=45)
for index, (x_frac, y_frac) in enumerate(beam_points, start=1):
    ax.text(x_frac * (width - 1), y_frac * (height - 1), str(index), color="white", ha="center", va="center")
ax.set_title("Random EDS beam positions")
ax.axis("off")
plt.tight_layout()


### Place the beam and collect EDS spectra


In [ ]:
spectra = []

for index, point in enumerate(beam_points, start=1):
    microscope.place_beam(point.tolist())
    key = microscope.acquire_spectrum("eds")
    node = client[key]["spectrum"]
    spectrum = np.squeeze(np.asarray(node.read()))
    metadata = dict(node.metadata)
    spectra.append({
        "point_index": index,
        "beam_position": point.tolist(),
        "tiled_key": key,
        "metadata": metadata,
        "spectrum": spectrum,
    })
    print(f"Point {index}: key={key}, beam={point.tolist()}, bins={spectrum.size}")


In [ ]:
fig, ax = plt.subplots(figsize=(8, 4))
for record in spectra:
    ax.plot(record["spectrum"], label=f"Point {record['point_index']}")
ax.set_xlabel("Channel")
ax.set_ylabel("Counts")
ax.set_title("EDS spectra from random beam positions")
ax.legend()
plt.tight_layout()


### Inspect EDS and microscope capabilities


In [ ]:
print('--- EDS attributes ---')
for attr in eds_proxy.get_attribute_list():
    print(f'  {attr}')

print('\n--- Microscope commands ---')
for cmd in mic_proxy.get_command_list():
    print(f'  {cmd}')

### Configure and acquire an EDS spectrum


In [ ]:
eds_proxy.exposure_time 

In [ ]:
key = microscope.acquire_spectrum("eds")
node = client[key]["spectrum"]
spectrum = np.squeeze(np.asarray(node.read()))
metadata = dict(node.metadata)

print("Tiled key     :", key)
print("Metadata      :", metadata)
print("Spectrum shape:", spectrum.shape)
print("Spectrum dtype:", spectrum.dtype)


### Plot the spectrum


In [ ]:
plt.figure(figsize=(8, 4))
plt.plot(spectrum)
plt.xlabel("Channel")
plt.ylabel("Counts")
plt.title("EDS spectrum")
plt.tight_layout()

In [ ]:
metadata

### Build a sidpy spectrum dataset


In [ ]:
dset = sidpy.Dataset.from_array(spectrum, name="EDS Spectrum")
dset.a.dimension_type = "SPECTRAL"
dset.data_type = "SPECTRUM"
dset.title = "EDS"
dset.set_dimension(0, sidpy.Dimension(np.arange(spectrum.shape[0]) * 20 - 120, quantity="Energy", units="eV"))
dset.quantity = "Intensity"
dset.units = "Counts"
dset.metadata = metadata

In [ ]:
view = dset.plot()